CẤU HÌNH KẾT NỐI & BIẾN MÔI TRƯỜNG

In [ ]:
import os
# --- CELL 1: CẤU HÌNH CHUNG (CHẠY 1 LẦN) ---

# 1. Thông tin Storage Account (Dùng lại của Notebook 1)
STORAGE_ACCOUNT_NAME = "nhatlong"
# HÃY DÁN KEY STORAGE (Key 1) CỦA BẠN VÀO DƯỚI ĐÂY:
STORAGE_ACCOUNT_KEY  = os.environ["AZURE_STORAGE_KEY"] 

# 2. Cấu hình Spark để có quyền Ghi vào Container Silver
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_KEY
)

# 3. Định nghĩa đường dẫn và Tên bảng (Để code bên dưới gọn gàng)
# Đường dẫn Checkpoint riêng cho lớp Silver (Không dùng chung với Bronze)
CHECKPOINT_SILVER_PATH = f"abfss://checkpoint@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/silver_telemetry"

# Tên các bảng (Table Names)
TBL_BRONZE_STREAM    = "bronze_server_telemetry"   # Nguồn luồng
TBL_BRONZE_STATIC    = "bronze_server_inventory"   # Nguồn tĩnh (để Join)
TBL_SILVER_OUTPUT    = "silver_server_telemetry"   # Đích đến

print(f"✅ Đã cấu hình xong! Kết nối tới Storage '{STORAGE_ACCOUNT_NAME}' thành công.")
print(f"   - Nguồn đọc: {TBL_BRONZE_STREAM} & {TBL_BRONZE_STATIC}")
print(f"   - Đích ghi : {TBL_SILVER_OUTPUT}")

ĐỌC DỮ LIỆU TỪ LỚP BRONZE

In [ ]:
# --- CELL 2: LOAD DỮ LIỆU BRONZE ---

# 1. Đọc bảng Tĩnh (Inventory) - Dùng để tra cứu vị trí Server
# Chúng ta cache (lưu vào RAM) bảng này vì nó nhỏ và dùng để Join liên tục
df_inventory = spark.read.table(TBL_BRONZE_STATIC).cache()

# 2. Đọc luồng Streaming (Telemetry) - Dữ liệu JSON thô
# Lưu ý: Dùng readStream để duy trì chế độ Real-time
df_stream_raw = spark.readStream.table(TBL_BRONZE_STREAM)

print("Đã load xong 2 DataFrame:")
print(f"1. df_inventory: {df_inventory.count()} dòng (Đã Cache)")
print("2. df_stream_raw: Đang chờ xử lý Streaming...")

TÁCH JSON VÀ LÀM SẠCH (TRANSFORMATION)

In [ ]:
# --- CELL 3: PARSE JSON & CLEANING ---
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql.functions import from_json, col, current_timestamp

# 1. Định nghĩa cấu trúc (Schema) của gói tin JSON gửi từ máy bạn lên
# Phải khớp với code Python Generator
json_schema = StructType([
    StructField("server_id", StringType(), True),
    StructField("timestamp", StringType(), True), # Lúc gửi là String ISO
    StructField("cpu_usage", DoubleType(), True),
    StructField("ram_usage", DoubleType(), True),
    StructField("temperature", IntegerType(), True),
    StructField("status", StringType(), True)
])

# 2. Tách cột JSON (Parsing)
df_parsed = df_stream_raw.select(
    # Hàm from_json biến chuỗi thành Struct, sau đó ta lấy từng cột ra
    from_json(col("raw_json"), json_schema).alias("data"), 
    col("event_time") # Giữ lại thời gian sự kiện gốc từ Event Hub
)

# 3. Làm phẳng (Flatten) và Ép kiểu
df_flattened = df_parsed.select(
    col("data.server_id"),
    col("data.timestamp").cast(TimestampType()).alias("generated_time"), # Chuyển String sang Time chuẩn
    col("data.cpu_usage"),
    col("data.ram_usage"),
    col("data.temperature"),
    col("data.status"),
    col("event_time").alias("ingestion_time")
)

# 4. LÀM SẠCH (Data Quality Check)
# Loại bỏ các dòng lỗi (Ví dụ: Generator thỉnh thoảng bắn lỗi CPU = -999)
df_clean = df_flattened.filter(
    (col("cpu_usage") >= 0) & 
    (col("cpu_usage") <= 100) &
    (col("server_id").isNotNull())
)

print("✅ Đã thiết lập xong Logic xử lý (Transformation Logic).")
# Lưu ý: Cell này chưa chạy ra dữ liệu ngay, nó chỉ mới "lên kế hoạch" (Lazy Evaluation).

BỔ SUNG THÔNG TIN (STREAM-STATIC JOIN)

In [ ]:
# --- CELL 4: JOIN VỚI BẢNG TĨNH (ENRICHMENT) ---

# Thực hiện Left Join: Giữ lại tất cả log Server, tìm thông tin bổ sung từ Inventory
df_silver_enriched = df_clean.join(
    df_inventory,
    on="server_id",       # Cột chung để ghép
    how="left"            # Left Join
)

# Chọn lọc và sắp xếp lại các cột cho đẹp
df_final_silver = df_silver_enriched.select(
    "server_id",
    "location",           # Cột mới lấy từ Inventory
    "admin_email",        # Cột mới lấy từ Inventory
    "data_center_id",     # Cột mới lấy từ Inventory
    "cpu_usage",
    "ram_usage",
    "temperature",
    "generated_time",
    "ingestion_time",
    current_timestamp().alias("processed_time") # Thêm dấu thời gian xử lý tại Silver
)

print("✅ Đã thiết lập xong Logic Join (Stream-Static Join).")

GHI VÀO LỚP SILVER (ACTION)

In [ ]:
# --- CELL 5: CHẠY STREAM VÀ GHI VÀO SILVER ---

# 1. Dọn dẹp Checkpoint cũ (Để tránh xung đột nếu chạy lại nhiều lần)
# Chỉ chạy dòng này khi bạn muốn reset luồng từ đầu. 
# dbutils.fs.rm(CHECKPOINT_SILVER_PATH, True) 

print(f"🚀 Đang khởi động luồng xử lý và ghi vào bảng '{TBL_SILVER_OUTPUT}'...")

# 2. WriteStream
query_silver = (df_final_silver.writeStream
    .format("delta")
    .outputMode("append")                 # Chỉ ghi thêm dữ liệu mới
    .option("checkpointLocation", CHECKPOINT_SILVER_PATH) # Nơi lưu trạng thái
    .option("mergeSchema", "true")        # Cho phép schema thay đổi linh hoạt
    .trigger(processingTime='5 seconds')  # Xử lý theo lô mỗi 5 giây
    .table(TBL_SILVER_OUTPUT)             # Tên bảng đích
)

print(f"Trạng thái: Luồng đang chạy! (ID: {query_silver.id})")

In [ ]:
%sql
-- Kiểm tra bảng Silver xem đã có dữ liệu sạch chưa
-- Bạn sẽ thấy các cột 'location', 'admin_email' đã xuất hiện!
SELECT * FROM silver_server_telemetry ORDER BY processed_time DESC LIMIT 10;